In [ ]:
import pandas as pd

# Conversor de Tempo
def converter_tempo_para_segundos(tempo_str):
    # Se a célula estiver vazia, ignora
    if pd.isna(tempo_str): 
        return None
    
    tempo_str = str(tempo_str).strip()
    
    # Se tiver os dois pontos (ex: 1:39.124)
    if ':' in tempo_str:
        minutos, segundos = tempo_str.split(':')
        # Garante que a vírgula brasileira vire ponto pro Python calcular
        segundos = segundos.replace(',', '.') 
        return round((float(minutos) * 60) + float(segundos), 3)
    else:
        # Se já for só segundos (raro em corrida, mas bom prevenir)
        return float(tempo_str.replace(',', '.'))

# O EXTRATOR PRINCIPAL 
def limpar_csv_stockcar(caminho_entrada, caminho_saida):
    print("Iniciando limpeza dos dados")

    # 1. Carrega o CSV bruto a jato (Motor C em ação)
    df = pd.read_csv(caminho_entrada, sep=',', decimal=',', encoding='utf-8-sig')

    # 2. Limpeza brutal dos nomes das colunas
    df.columns = df.columns.str.strip().str.replace('"', '')

    # 3. Cria a coluna do Piloto
    df['Piloto'] = None 

    # 4. Identifica e preenche o nome do piloto
    mascara_piloto = df['Lap'].isna()
    df.loc[mascara_piloto, 'Piloto'] = df.loc[mascara_piloto, 'Time of Day']
    df['Piloto'] = df['Piloto'].ffill()

    # 5. Remove as linhas de cabeçalho
    df_limpo = df.dropna(subset=['Lap']).copy()

    # 6. Reorganiza as colunas
    colunas = df_limpo.columns.tolist()
    colunas.insert(0, colunas.pop(colunas.index('Piloto')))
    df_limpo = df_limpo[colunas]

    # Convertendo o Tempo de Volta para Segundos 
    print("Convertendo os tempos de volta para segundos absolutos")
    df_limpo['Lap Tm (Segundos)'] = df_limpo['Lap Tm'].apply(converter_tempo_para_segundos)
    
    # Move a nova coluna para ficar logo do lado do tempo original
    colunas = df_limpo.columns.tolist()
    colunas.insert(colunas.index('Lap Tm') + 1, colunas.pop(colunas.index('Lap Tm (Segundos)')))
    df_limpo = df_limpo[colunas]

    # 7. Salva o CSV final
    df_limpo.to_csv(caminho_saida, index=False, sep=';', decimal=',')
    print(f"Limpeza concluída! Base de dados perfeita salva em: {caminho_saida}")

# --- ÁREA DE EXECUÇÃO PARA O TREINO 2 ---
arquivo_bruto_t2 = '../data/01_raw/voltas_curvelo_t2.csv' 
arquivo_pronto_t2 = '../data/02_interim/dados_limpos_T2.csv'

limpar_csv_stockcar(arquivo_bruto_t2, arquivo_pronto_t2)

🔧 Levando o carro para a garagem: Iniciando limpeza dos dados...
⏱️ Convertendo os tempos de volta para segundos absolutos...
🏁 Limpeza concluída! Base de dados perfeita salva em: ../data/02_interim/dados_limpos_T2.csv
